## 0. Install Dependencies

In [1]:
!pip install torch torchvision transformers open-clip-torch
!pip install hnswlib pillow tqdm numpy scikit-learn
!pip install ultralytics        # YOLOv8


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp312-cp312-linux_x86_64.whl size=2733576 sha256=e9fe3a770594c1c31aff7c9c18649e05088edda71e84be8dc63869726a5f1682
  Stored in directory: /root/.cache/pip/wheels/ac/39/b3/cbd7f9cbb76501d2d5fbc84956e70d0b94e788aac87bda465e
Successfully built hnswlib
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 48.0 MB/s eta 0:00:00


## 1. Configuration

**Config A — Vision-only CLIP (α = 1)**

This is the baseline from the ablation study:
- CLIP vision encoder: **frozen** (pre-trained weights, no updates)
- CLIP text encoder: **not used** (α = 1, so text embedding is zeroed out)
- BLIP-2: **not used** (no captioning in Config A)
- YOLO: **frozen** (product localisation only)

The embedding formula collapses to pure visual:
`v_i = CLIP_visual(crop)`  (α = 1 → text term vanishes)

**Purpose:** Baseline with no semantic alignment and no fine-tuning.


In [2]:
import os, json, random
import numpy as np
import torch

# ── PATHS ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = "/kaggle/input/datasets/suhasnagabandi/deepfashionset"
IMG_ROOT    = os.path.join(DATA_ROOT, "img", "img")
BBOX_FILE   = os.path.join(DATA_ROOT, "list_bbox_inshop.txt")
EVAL_FILE   = os.path.join(DATA_ROOT, "list_eval_partition.txt")
DESC_FILE   = os.path.join(DATA_ROOT, "list_description_inshop.json")
CKPT_DIR    = "./checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# ── EXPERIMENT SETTINGS ───────────────────────────────────────────────────────
SEEDS   = [599, 600, 124, 605]
ALPHA   = 1.0        # Config A: vision-only, text encoder not used
TOP_K   = [5, 10, 15]

# ── YOLO SETTINGS ─────────────────────────────────────────────────────────────
YOLO_MODEL_NAME = "yolov8n.pt"
YOLO_CONF       = 0.25
YOLO_CROP_CACHE = os.path.join(CKPT_DIR, "yolo_crop_cache.pkl")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Config A — Vision-only CLIP (α = {ALPHA}), fully frozen, no BLIP-2")


Device: cuda
Config A — Vision-only CLIP (α = 1.0), fully frozen, no BLIP-2


## 2. Data Loading & Parsing

In [3]:
# ── Parse eval partition ──────────────────────────────────────────────────────
def parse_eval_partition(path):
    train_imgs, query_imgs, gallery_imgs = [], [], []
    with open(path) as f:
        lines = f.read().splitlines()
    # first two lines: total count + header
    for line in lines[2:]:
        parts = line.split()
        if len(parts) < 3:
            continue
        img_path, item_id, split = parts[0], parts[1], parts[2]
        record = {"img": img_path, "item_id": item_id}
        if split == "train":
            train_imgs.append(record)
        elif split == "query":
            query_imgs.append(record)
        elif split == "gallery":
            gallery_imgs.append(record)
    return train_imgs, query_imgs, gallery_imgs

train_imgs, query_imgs, gallery_imgs = parse_eval_partition(EVAL_FILE)
print(f"Train: {len(train_imgs)} | Query: {len(query_imgs)} | Gallery: {len(gallery_imgs)}")

# ── Parse bounding boxes ──────────────────────────────────────────────────────
def parse_bbox(path):
    bbox_map = {}
    with open(path) as f:
        lines = f.read().splitlines()
    for line in lines[2:]:
        parts = line.split()
        if len(parts) < 7:
            continue
        img_path = parts[0]
        x1, y1, x2, y2 = int(parts[3]), int(parts[4]), int(parts[5]), int(parts[6])
        bbox_map[img_path] = (x1, y1, x2, y2)
    return bbox_map

bbox_map = parse_bbox(BBOX_FILE)
print(f"BBox entries: {len(bbox_map)}")

# ── Parse descriptions (kept for fallback caption compatibility) ───────────────
with open(DESC_FILE) as f:
    desc_raw = json.load(f)
desc_map = {}
for entry in desc_raw:
    item_id = entry["item"]
    color = entry.get("color", "")
    descs = entry.get("description", [])
    text = f"{color}. " + " ".join(descs[:2]) if descs else color
    desc_map[item_id] = text[:300]
print(f"Descriptions loaded: {len(desc_map)}")


Train: 25882 | Query: 14218 | Gallery: 12612
BBox entries: 52712
Descriptions loaded: 7982


## 2b. Load YOLO — Product Localisation

YOLO crops the primary clothing item before passing to CLIP (Step 1 in the PDF pipeline).
Stays **frozen** throughout. Results cached to avoid re-running.


In [4]:
from ultralytics import YOLO
import pickle
from PIL import Image

# ── Load YOLO (frozen — inference only) ───────────────────────────────────────
yolo_model = YOLO(YOLO_MODEL_NAME)
print(f"YOLO model loaded: {YOLO_MODEL_NAME}")


def yolo_crop(img_pil, yolo_model, conf=YOLO_CONF):
    import numpy as np
    img_np = np.array(img_pil)
    results = yolo_model(img_np, conf=conf, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return img_pil, None
    best_idx = int(boxes.conf.argmax())
    x1, y1, x2, y2 = boxes.xyxy[best_idx].cpu().numpy().astype(int)
    w, h = img_pil.size
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 > x1 and y2 > y1:
        return img_pil.crop((x1, y1, x2, y2)), (x1, y1, x2, y2)
    return img_pil, None


def build_yolo_crop_cache(records, img_root, bbox_map, yolo_model,
                           conf=YOLO_CONF, cache_path=YOLO_CROP_CACHE):
    if os.path.exists(cache_path):
        print(f"Loading YOLO crop cache: {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print(f"Running YOLO on {len(records)} images — this runs once and is cached …")
    crop_box_cache = {}
    for i, r in enumerate(records):
        img_path  = r["img"]
        full_path = os.path.join(img_root, img_path)
        try:
            img = Image.open(full_path).convert("RGB")
            _, yolo_box = yolo_crop(img, yolo_model, conf)
            crop_box_cache[img_path] = yolo_box
        except Exception:
            crop_box_cache[img_path] = None
        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(records)} images processed")

    with open(cache_path, "wb") as f:
        pickle.dump(crop_box_cache, f)
    print(f"YOLO crop cache saved → {cache_path}")
    return crop_box_cache


def get_cropped_image(img_pil, img_path, bbox_map, yolo_box_cache):
    """Priority: cached YOLO box → GT bbox → full image."""
    yolo_box = yolo_box_cache.get(img_path)
    if yolo_box is not None:
        x1, y1, x2, y2 = yolo_box
        w, h = img_pil.size
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 > x1 and y2 > y1:
            return img_pil.crop((x1, y1, x2, y2))
    gt_bbox = bbox_map.get(img_path)
    if gt_bbox is not None:
        x1, y1, x2, y2 = gt_bbox
        w, h = img_pil.size
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 > x1 and y2 > y1:
            return img_pil.crop((x1, y1, x2, y2))
    return img_pil


all_records_for_cache = train_imgs + gallery_imgs + query_imgs
yolo_box_cache = build_yolo_crop_cache(
    all_records_for_cache, IMG_ROOT, bbox_map, yolo_model
)

detected     = sum(1 for v in yolo_box_cache.values() if v is not None)
total_cached = len(yolo_box_cache)
print(f"YOLO detection rate: {detected}/{total_cached} "
      f"({100*detected/max(total_cached,1):.1f}%) — rest fall back to GT bbox or full image")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO model loaded: yolov8n.pt
Running YOLO on 52712 images — this runs once and is cached …
requirements: Ultralytics requirement ['pi-heif'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 225ms
 Downloaded pi-heif
Prepared 1 package in 65ms
Installed 1 package in 5ms
 + pi-heif==1.3.0

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

  1000/52712 images processed
  2000/52712 images processed
  3000/52712 images processed
  4000/52712 images processed
  5000/52712 images processed
  6000/52712 images processed
  7000/52712 images processed
  8000/5271

## 3. Load CLIP (Fully Frozen)

**Config A uses vision encoder only.**
- ALL parameters frozen — no fine-tuning.
- Text encoder is loaded but never called (α = 1 means text weight = 0).
- Identical to Config B loading, but α is hardcoded to 1.0.


In [5]:
import open_clip
import torchvision.transforms as T

def load_clip_model_frozen():
    """Load CLIP with ALL parameters frozen — Config A baseline."""
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model = model.to(DEVICE)

    # Freeze ALL parameters — no fine-tuning in Config A
    for p in model.parameters():
        p.requires_grad = False
    model.eval()

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} (0% — fully frozen)")
    return model, tokenizer


clip_model, clip_tokenizer = load_clip_model_frozen()

CLIP_IMG_SIZE = 224
clip_transform = T.Compose([
    T.Resize((CLIP_IMG_SIZE, CLIP_IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711]),
])
print("CLIP loaded (fully frozen) and transform defined.")


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Trainable params: 0 / 151,277,313 (0% — fully frozen)
CLIP loaded (fully frozen) and transform defined.


## 4. Vision-Only Embedding Extraction + HNSW Indexing

Config A embedding formula (α = 1):
`v_i = CLIP_visual(crop)`  — text encoder not called at all.

HNSW settings are **identical to Config C**: `M=32, ef_construction=200, set_ef=64`
to ensure a fair comparison across all three configs.


In [6]:
import hnswlib
import torch.nn.functional as F
from tqdm.notebook import tqdm


def extract_visual_embeddings(records, clip_model, img_root, bbox_map,
                               yolo_box_cache, batch_size=64):
    """
    Config A — Vision-only embedding (α = 1).
    Offline indexing Steps 1 + 3 (PDF):
      Step 1: YOLO crop  (via cached yolo_box_cache)
      Step 3: CLIP visual embedding only — text encoder not used
        v_i = CLIP_visual(crop),  normalised to unit length
    Returns:
      embeddings : (N, D) numpy, L2-normalised
      img_paths  : list[str]
      item_ids   : list[str]
    """
    clip_model.eval()
    all_embeds, all_paths, all_items = [], [], []

    for start in tqdm(range(0, len(records), batch_size),
                      desc="Extracting visual embeddings (α=1)"):
        batch = records[start:start + batch_size]
        imgs_tensors  = []
        valid_records = []

        for r in batch:
            full_path = os.path.join(img_root, r["img"])
            try:
                img = Image.open(full_path).convert("RGB")
                # Step 1: YOLO product localisation
                img = get_cropped_image(img, r["img"], bbox_map, yolo_box_cache)
                imgs_tensors.append(clip_transform(img))
            except Exception:
                imgs_tensors.append(torch.zeros(3, 224, 224))
            valid_records.append(r)

        imgs_batch = torch.stack(imgs_tensors).to(DEVICE)

        with torch.no_grad():
            # Step 3: vision-only CLIP embedding (text encoder not called)
            vis_emb = clip_model.encode_image(imgs_batch)
            vis_emb = F.normalize(vis_emb.float(), dim=-1)   # unit norm

        all_embeds.append(vis_emb.cpu().numpy())
        all_paths.extend([r["img"] for r in valid_records])
        all_items.extend([r["item_id"] for r in valid_records])

    return np.vstack(all_embeds), all_paths, all_items


def build_hnsw_index(embeddings, dim, ef_construction=200, M=32):
    """
    Step 4 (PDF): build HNSW ANN index.
    Same settings as Config C: M=32, ef_construction=200, set_ef=64.
    """
    index = hnswlib.Index(space="cosine", dim=dim)
    index.init_index(max_elements=len(embeddings),
                     ef_construction=ef_construction, M=M)
    index.add_items(embeddings, list(range(len(embeddings))))
    index.set_ef(64)
    return index


print("Embedding + HNSW functions defined.")
print(f"HNSW settings: M=32, ef_construction=200, set_ef=64 (identical to Config C)")


Embedding + HNSW functions defined.
HNSW settings: M=32, ef_construction=200, set_ef=64 (identical to Config C)


## 5. Retrieval Metrics

In [7]:
from collections import Counter

def build_gallery_item_counts(gallery_item_ids):
    """True count of relevant gallery images per item_id."""
    return dict(Counter(gallery_item_ids))


def recall_at_k(query_item_ids, ranked_gallery_item_ids, k, gallery_item_counts):
    """
    Recall@K = fraction of queries where >=1 relevant item is in top-K.
    Binary hit metric.
    """
    hits = 0
    for q_id, ranked in zip(query_item_ids, ranked_gallery_item_ids):
        if any(r_id == q_id for r_id in ranked[:k]):
            hits += 1
    return hits / len(query_item_ids)


def ndcg_at_k(query_item_ids, ranked_gallery_item_ids, k, gallery_item_counts):
    """
    NDCG@K — non-decreasing with K.
    IDCG uses min(n_rel, k) so NDCG@5 <= NDCG@10 always holds.
    """
    scores = []
    for q_id, ranked in zip(query_item_ids, ranked_gallery_item_ids):
        dcg = sum(
            1.0 / np.log2(i + 2)
            for i, r_id in enumerate(ranked[:k])
            if r_id == q_id
        )
        n_rel   = gallery_item_counts.get(q_id, 0)
        ideal_n = min(n_rel, k)
        idcg    = sum(1.0 / np.log2(i + 2) for i in range(ideal_n))
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return float(np.mean(scores))


def map_at_k(query_item_ids, ranked_gallery_item_ids, k, gallery_item_counts):
    """
    mAP@K — non-decreasing with K.
    Normalised by min(n_rel, k) to prevent decreasing mAP at higher K.
    """
    aps = []
    for q_id, ranked in zip(query_item_ids, ranked_gallery_item_ids):
        top_k    = ranked[:k]
        hits     = 0
        prec_sum = 0.0
        for i, r_id in enumerate(top_k):
            if r_id == q_id:
                hits     += 1
                prec_sum += hits / (i + 1)
        n_rel = gallery_item_counts.get(q_id, 0)
        denom = min(n_rel, k)
        aps.append(prec_sum / denom if denom > 0 else 0.0)
    return float(np.mean(aps))


def compute_all_metrics(query_item_ids, ranked_gallery_item_ids,
                         gallery_item_ids, ks=TOP_K):
    gallery_item_counts = build_gallery_item_counts(gallery_item_ids)
    results = {}
    for k in ks:
        results[f"Recall@{k}"] = recall_at_k(
            query_item_ids, ranked_gallery_item_ids, k, gallery_item_counts)
        results[f"NDCG@{k}"]   = ndcg_at_k(
            query_item_ids, ranked_gallery_item_ids, k, gallery_item_counts)
        results[f"mAP@{k}"]    = map_at_k(
            query_item_ids, ranked_gallery_item_ids, k, gallery_item_counts)
    return results


print("Metric functions defined — identical to Config B and C for fair comparison.")


Metric functions defined — identical to Config B and C for fair comparison.


## 6. Full Evaluation Loop — Config A

For each seed:
1. Extract vision-only gallery embeddings (YOLO crop → CLIP visual) → HNSW index
2. Extract vision-only query embeddings
3. Retrieve top-15 from gallery via cosine similarity
4. Compute Recall@K, NDCG@K, mAP@K for K ∈ {5, 10, 15}

**Note:** α = 1.0 is fixed — no α sweep needed for Config A per the PDF spec.
The seed loop rebuilds the HNSW graph (which has internal randomness) to
give an honest variance estimate across seeds.


In [8]:
all_results_A = {}   # seed -> metrics dict

for seed in SEEDS:
    print(f"\n--- Evaluating: Config A | α=1.0 (vision-only) | seed={seed} ---")

    # Set random state for HNSW graph construction reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # 1. Gallery embeddings — vision-only CLIP (Steps 1 + 3 offline pipeline)
    gal_embeds, gal_paths, gal_item_ids = extract_visual_embeddings(
        gallery_imgs, clip_model, IMG_ROOT, bbox_map, yolo_box_cache
    )
    dim = gal_embeds.shape[1]

    # 2. Build HNSW index (Step 4 — same settings as Config C)
    index = build_hnsw_index(gal_embeds, dim)

    # 3. Query embeddings — vision-only (Steps 1-2 online query pipeline)
    q_embeds, q_paths, q_item_ids = extract_visual_embeddings(
        query_imgs, clip_model, IMG_ROOT, bbox_map, yolo_box_cache
    )

    # 4. ANN retrieval — top-15 candidates (Step 3 online query pipeline)
    labels_batch, _ = index.knn_query(q_embeds, k=min(15, len(gal_embeds)))
    ranked_gallery_item_ids = [
        [gal_item_ids[idx] for idx in row]
        for row in labels_batch
    ]

    # 5. Compute metrics
    metrics = compute_all_metrics(q_item_ids, ranked_gallery_item_ids, gal_item_ids)
    all_results_A[seed] = metrics

    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

print("\n✓ All Config A evaluations complete.")



--- Evaluating: Config A | α=1.0 (vision-only) | seed=599 ---


Extracting visual embeddings (α=1):   0%|          | 0/198 [00:00<?, ?it/s]

Extracting visual embeddings (α=1):   0%|          | 0/223 [00:00<?, ?it/s]

  Recall@5: 0.0006
  NDCG@5: 0.0006
  mAP@5: 0.0006
  Recall@10: 0.0012
  NDCG@10: 0.0007
  mAP@10: 0.0005
  Recall@15: 0.0028
  NDCG@15: 0.0008
  mAP@15: 0.0006

--- Evaluating: Config A | α=1.0 (vision-only) | seed=600 ---


Extracting visual embeddings (α=1):   0%|          | 0/198 [00:00<?, ?it/s]

...

Extracting visual embeddings (α=1):   0%|          | 0/223 [00:00<?, ?it/s]

  Recall@5: 0.0008
  NDCG@5: 0.0006
  mAP@5: 0.0005
  Recall@10: 0.0013
  NDCG@10: 0.0007
  mAP@10: 0.0005
  Recall@15: 0.0017
  NDCG@15: 0.0008
  mAP@15: 0.0006

--- Evaluating: Config A | α=1.0 (vision-only) | seed=605 ---


Extracting visual embeddings (α=1):   0%|          | 0/198 [00:00<?, ?it/s]

Extracting visual embeddings (α=1):   0%|          | 0/223 [00:00<?, ?it/s]

  Recall@5: 0.0006
  NDCG@5: 0.0006
  mAP@5: 0.0006
  Recall@10: 0.0012
  NDCG@10: 0.0007
  mAP@10: 0.0005
  Recall@15: 0.0025
  NDCG@15: 0.0008
  mAP@15: 0.0006

✓ All Config A evaluations complete.


## 7. Aggregate Results: Mean ± Std Across Seeds

In [9]:
import pandas as pd

metric_keys = [f"{m}@{k}" for m in ["Recall", "NDCG", "mAP"] for k in TOP_K]
seed_metrics = {key: [] for key in metric_keys}

for seed in SEEDS:
    m = all_results_A[seed]
    for key in metric_keys:
        seed_metrics[key].append(m[key])

row = {"Config": "A (Vision-only CLIP)", "α": 1.0}
for key in metric_keys:
    vals = seed_metrics[key]
    row[key] = f"{np.mean(vals):.4f} ± {np.std(vals):.4f}"

df = pd.DataFrame([row]).set_index(["Config", "α"])
print("\n=== Ablation Study — Config A Results ===")
print(df.to_string())
df



=== Ablation Study — Config A Results ===
                                 Recall@5        Recall@10        Recall@15           NDCG@5          NDCG@10          NDCG@15            mAP@5           mAP@10           mAP@15
Config               α                                                                                                                                                           
A (Vision-only CLIP) 1.0  0.0007 ± 0.0001  0.0013 ± 0.0001  0.0022 ± 0.0005  0.0006 ± 0.0000  0.0007 ± 0.0000  0.0008 ± 0.0000  0.0006 ± 0.0001  0.0005 ± 0.0000  0.0006 ± 0.0000


,,Recall@5,Recall@10,Recall@15,NDCG@5,NDCG@10,NDCG@15,mAP@5,mAP@10,mAP@15
Config,α,,,,,,,,,
A (Vision-only CLIP),1.0,0.0007 ± 0.0001,0.0013 ± 0.0001,0.0022 ± 0.0005,0.0006 ± 0.0000,0.0007 ± 0.0000,0.0008 ± 0.0000,0.0006 ± 0.0001,0.0005 ± 0.0000,0.0006 ± 0.0000


## 8. Save Results to CSV

In [10]:
# Per-seed raw results
raw_rows = []
for seed, metrics in all_results_A.items():
    row = {"Config": "A", "alpha": 1.0, "seed": seed}
    row.update(metrics)
    raw_rows.append(row)

df_raw = pd.DataFrame(raw_rows)
df_raw.to_csv("ablation_A_raw_results.csv", index=False)
df.to_csv("ablation_A_summary.csv")
print("Saved: ablation_A_raw_results.csv and ablation_A_summary.csv")


Saved: ablation_A_raw_results.csv and ablation_A_summary.csv
